# Week 9 Lab: Breaking and Defending Face Detection

## Lab Overview

Today we'll explore how face detection models can be fooled and defended. You'll see firsthand how small changes can cause AI systems to fail, understand why this happens, and experiment with simple defense strategies.

**Learning Objectives:**
- Witness how robust (or fragile) face detection really is
- Understand what makes models vulnerable
- Experiment with attacks and defenses

**What You'll Need:**
- Webcam access
- Printable paper (optional but recommended)
- Markers or face paint (optional for brave students!)

### Haar Cascade
For today's lab, you will see a Haar classifier, or a Haar cascade classifier. It is a machine learning object detection program that identifies objects in an image and video. Haar Cascade is a simple and fast method used in computer vision to detect objects, especially faces, in images or videos. It works by looking for specific patterns of light and dark areas (called "features") in small parts of an image. By scanning the image with these features, it can quickly find where faces or other objects are located. It’s like using a set of simple rules to spot faces based on common shapes and edges. You can read more about it [here](https://medium.com/analytics-vidhya/haar-cascades-explained-38210e57970d)


## Setup and Imports

First, let's import all the libraries we'll need for this lab.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display, Image
import ipywidgets as widgets
from io import BytesIO
import PIL.Image
import time

# Set up matplotlib for better display in notebook
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 80

print("Libraries loaded successfully!")

## Part 1: Setting Up Our Face Detection System

First, let's establish our baseline face detection system and see how well it works under normal conditions.

In [ ]:
# Load face detection model
# We'll use Haar Cascades

# Try different methods to load the cascade classifier
cascade_loaded = False

# Method 1: Try using cv2.data (works in newer OpenCV versions)
try:
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    cascade_loaded = True
    print("Loaded using cv2.data.haarcascades")
except AttributeError:
    pass

# Method 2: Try local file
if not cascade_loaded:
    try:
        face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')
        if not face_cascade.empty():
            cascade_loaded = True
            print("Loaded from local file")
    except:
        pass

# Method 3: Download if needed
if not cascade_loaded:
    import os
    import urllib.request
    
    cascade_url = 'https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml'
    cascade_file = 'haarcascade_frontalface_default.xml'
    
    if not os.path.exists(cascade_file):
        print("Downloading cascade file...")
        urllib.request.urlretrieve(cascade_url, cascade_file)
    
    face_cascade = cv2.CascadeClassifier(cascade_file)
    print("Downloaded and loaded cascade file")

# Verify cascade loaded properly
if face_cascade.empty():
    print("ERROR: Could not load face cascade classifier!")
else:
    print("Face cascade loaded successfully.")

def detect_faces_cascade(image, scale_factor=1.1, min_neighbors=5):
    """
    Detect faces using Haar Cascade
    Returns: list of (x, y, w, h) tuples for each face
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=scale_factor, minNeighbors=min_neighbors)
    return faces

def draw_faces(image, faces, color=(0, 255, 0), thickness=2):
    """
    Draw rectangles around detected faces
    """
    image_copy = image.copy()
    for (x, y, w, h) in faces:
        cv2.rectangle(image_copy, (x, y), (x+w, y+h), color, thickness)
        # Add confidence text (Haar doesn't give confidence, so we'll simulate)
        cv2.putText(image_copy, "Face", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    return image_copy

print("\nFace detection system ready.")

### Task 1.1: Capture Baseline Image

Let's capture a baseline image from your webcam where face detection works well.

In [ ]:
# Function to capture image from webcam
def capture_single_frame():
    """Capture a single frame from webcam"""
    cap = cv2.VideoCapture(0)
    
    # Give camera time to initialize
    time.sleep(1)
    
    # Capture a few frames to let camera adjust
    for _ in range(5):
        ret, frame = cap.read()
    
    # Capture the actual frame
    ret, frame = cap.read()
    cap.release()
    
    if ret:
        return frame
    else:
        print("Failed to capture frame")
        return None

# Capture with 3-second countdown
print("Capturing image in 3 seconds... Get ready!")
for i in range(3, 0, -1):
    print(f"{i}...")
    time.sleep(1)

print("Capturing NOW...")
baseline_image = capture_single_frame()

if baseline_image is not None:
    faces = detect_faces_cascade(baseline_image)
    display_img = draw_faces(baseline_image, faces)
    
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(display_img, cv2.COLOR_BGR2RGB))
    plt.title(f'Baseline Image - {len(faces)} face(s) detected')
    plt.axis('off')
    plt.show()
    
    print(f"\nGreat, you have detected {len(faces)} face(s).")
else:
    print("Failed to capture. Creating a default test image...")
    baseline_image = np.ones((480, 640, 3), dtype=np.uint8) * 200

### Task 1.2: Sensitivity Testing

Let's test how different parameter settings affect face detection sensitivity.

**What the Parameters Do:**

`scale_factor (default 1.1)`:
- Controls how much the detector shrinks the image at each step
- Lower values (1.05) = more thorough search = finds smaller/more faces but slower
- Higher values (1.3) = faster but might miss faces

`min_neighbors (default 5)`:
- How many overlapping detections needed to confirm it's a face
- Lower values (3) = more sensitive, detects more "faces" (including false positives)
- Higher values (9) = more strict, only very confident detections

In [ ]:
# Test different parameter values
scale_factors = [1.05, 1.1, 1.2, 1.3]
min_neighbors_values = [3, 5, 7, 9]

# Create a grid of results
fig, axes = plt.subplots(len(scale_factors), len(min_neighbors_values), figsize=(16, 16))

for i, scale in enumerate(scale_factors):
    for j, neighbors in enumerate(min_neighbors_values):
        faces = detect_faces_cascade(baseline_image, scale, neighbors)
        result_img = draw_faces(baseline_image, faces)
        
        axes[i, j].imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
        axes[i, j].set_title(f'Scale: {scale}, Neighbors: {neighbors}\nFaces: {len(faces)}')
        axes[i, j].axis('off')

plt.suptitle('Face Detection Sensitivity Analysis', fontsize=16)
plt.tight_layout()
plt.show()

# Interactive parameter testing
@widgets.interact(scale_factor=(1.01, 1.5, 0.01), min_neighbors=(1, 20, 1))
def test_parameters(scale_factor=1.1, min_neighbors=5):
    faces = detect_faces_cascade(baseline_image, scale_factor, min_neighbors)
    result_img = draw_faces(baseline_image, faces)
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    plt.title(f'Detected {len(faces)} face(s)')
    plt.axis('off')
    plt.show()

## Part 2: Simple Perturbations That Break Detection

Now let's see how simple image modifications can fool the detector.

In [ ]:
def add_noise(image, noise_type='gaussian', intensity=0.1):
    """
    Add different types of noise to an image
    """
    if noise_type == 'gaussian':
        noise = np.random.randn(*image.shape) * 255 * intensity
        noisy = image + noise
        
    elif noise_type == 'salt_pepper':
        noisy = image.copy()
        num_pixels = int(intensity * image.size)
        # Salt
        coords = [np.random.randint(0, i-1, int(num_pixels/2)) for i in image.shape[:2]]
        noisy[coords[0], coords[1], :] = 255
        # Pepper
        coords = [np.random.randint(0, i-1, int(num_pixels/2)) for i in image.shape[:2]]
        noisy[coords[0], coords[1], :] = 0
        
    elif noise_type == 'uniform':
        noise = np.random.uniform(-255*intensity, 255*intensity, image.shape)
        noisy = image + noise
    
    return np.clip(noisy, 0, 255).astype(np.uint8)

def adjust_brightness_contrast(image, brightness=0, contrast=1):
    """
    Adjust brightness and contrast
    brightness: -100 to 100
    contrast: 0.5 to 3.0
    """
    adjusted = cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)
    return adjusted

print("Noise and adjustment functions ready!")

### Task 2.1: Noise Experiments

Test how different types and levels of noise affect face detection.

In [ ]:
# Test different noise levels
noise_levels = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, noise_level in enumerate(noise_levels):
    if noise_level == 0:
        noisy = baseline_image
    else:
        noisy = add_noise(baseline_image, 'gaussian', noise_level)
    
    faces = detect_faces_cascade(noisy)
    result_img = draw_faces(noisy, faces)
    
    axes[i].imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f'Noise Level: {noise_level}\nFaces: {len(faces)}')
    axes[i].axis('off')

plt.suptitle('Effect of Gaussian Noise on Face Detection', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Compare different noise types
noise_types = ['gaussian', 'salt_pepper', 'uniform']
noise_intensity = 0.1

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Original
faces_orig = detect_faces_cascade(baseline_image)
axes[0].imshow(cv2.cvtColor(draw_faces(baseline_image, faces_orig), cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Original\nFaces: {len(faces_orig)}')
axes[0].axis('off')

# Different noise types
for i, noise_type in enumerate(noise_types):
    noisy = add_noise(baseline_image, noise_type, noise_intensity)
    faces = detect_faces_cascade(noisy)
    result_img = draw_faces(noisy, faces)
    
    axes[i+1].imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    axes[i+1].set_title(f'{noise_type.title()}\nFaces: {len(faces)}')
    axes[i+1].axis('off')

plt.suptitle('Different Noise Types (Intensity: 0.1)', fontsize=16)
plt.tight_layout()
plt.show()

### Task 2.2: Environmental Changes

Test how brightness, contrast, and blur affect detection.

In [ ]:
# Test environmental modifications
modifications = [
    ('Original', baseline_image),
    ('Brightness +50', adjust_brightness_contrast(baseline_image, brightness=50)),
    ('Brightness -50', adjust_brightness_contrast(baseline_image, brightness=-50)),
    ('Contrast 0.5', adjust_brightness_contrast(baseline_image, contrast=0.5)),
    ('Contrast 2.0', adjust_brightness_contrast(baseline_image, contrast=2.0)),
    ('Gaussian Blur', cv2.GaussianBlur(baseline_image, (15, 15), 0)),
    ('Motion Blur', cv2.filter2D(baseline_image, -1, np.ones((1, 15))/15)),
    ('Box Blur', cv2.boxFilter(baseline_image, -1, (10, 10)))
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, (name, modified) in enumerate(modifications):
    faces = detect_faces_cascade(modified)
    result_img = draw_faces(modified, faces)
    
    axes[i].imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f'{name}\nFaces: {len(faces)}')
    axes[i].axis('off')

plt.suptitle('Environmental Modifications Effect on Face Detection', fontsize=16)
plt.tight_layout()
plt.show()

## Part 3: Pattern-Based Attacks

Let's create specific patterns designed to confuse face detection.

In [ ]:
def create_grid_pattern(shape, grid_size=20, intensity=0.3):
    """
    Create a grid pattern overlay
    """
    pattern = np.zeros(shape, dtype=np.uint8)
    pattern[::grid_size, :] = 255 * intensity
    pattern[:, ::grid_size] = 255 * intensity
    return pattern

def create_diagonal_stripes(shape, stripe_width=10, intensity=0.3):
    """
    Create diagonal stripe pattern
    """
    pattern = np.zeros(shape[:2], dtype=np.uint8)
    h, w = shape[:2]
    
    for i in range(-h, w, stripe_width * 2):
        cv2.line(pattern, (i, 0), (i + h, h), int(255 * intensity), stripe_width)
    
    return np.stack([pattern] * 3, axis=-1)

def create_dot_pattern(shape, dot_spacing=30, dot_size=5, intensity=0.3):
    """
    Create a dot pattern
    """
    pattern = np.zeros(shape, dtype=np.uint8)
    for y in range(0, shape[0], dot_spacing):
        for x in range(0, shape[1], dot_spacing):
            cv2.circle(pattern, (x, y), dot_size, (255 * intensity,) * 3, -1)
    return pattern

def create_noise_patches(image, faces, patch_size=0.3):
    """
    Add noise patches to face regions
    """
    result = image.copy()
    for (x, y, w, h) in faces:
        # Calculate patch position (center of face)
        patch_w = int(w * patch_size)
        patch_h = int(h * patch_size)
        patch_x = x + (w - patch_w) // 2
        patch_y = y + (h - patch_h) // 2
        
        # Add noise patch
        noise_patch = np.random.randint(0, 100, (patch_h, patch_w, 3))
        result[patch_y:patch_y+patch_h, patch_x:patch_x+patch_w] += noise_patch.astype(np.uint8)
    
    return np.clip(result, 0, 255).astype(np.uint8)

print("Pattern generation functions ready.")

### Task 3.1: Pattern Effectiveness

Test which patterns are most effective at breaking face detection.

In [ ]:
# Test different patterns
patterns = [
    ('Original', baseline_image),
    ('Grid', cv2.addWeighted(baseline_image, 0.8, 
                            create_grid_pattern(baseline_image.shape), 0.2, 0)),
    ('Diagonal Stripes', cv2.addWeighted(baseline_image, 0.8,
                                        create_diagonal_stripes(baseline_image.shape), 0.2, 0)),
    ('Dots', cv2.addWeighted(baseline_image, 0.8,
                            create_dot_pattern(baseline_image.shape), 0.2, 0)),
    ('Noise Patch', create_noise_patches(baseline_image, detect_faces_cascade(baseline_image)))
]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for i, (name, pattern_img) in enumerate(patterns):
    faces = detect_faces_cascade(pattern_img)
    result_img = draw_faces(pattern_img, faces)
    
    axes[i].imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f'{name}\nFaces: {len(faces)}')
    axes[i].axis('off')

plt.suptitle('Pattern-Based Attacks on Face Detection', fontsize=16)
plt.tight_layout()
plt.show()

### Task 3.2: Interactive Pattern Testing

Experiment with different pattern parameters.

In [ ]:
@widgets.interact(
    pattern_type=['None', 'Grid', 'Stripes', 'Dots'],
    intensity=(0.0, 1.0, 0.1),
    size=(5, 50, 5)
)
def test_patterns(pattern_type='None', intensity=0.3, size=20):
    if pattern_type == 'None':
        result_img = baseline_image
    elif pattern_type == 'Grid':
        pattern = create_grid_pattern(baseline_image.shape, grid_size=size, intensity=intensity)
        result_img = cv2.addWeighted(baseline_image, 1-intensity, pattern, intensity, 0)
    elif pattern_type == 'Stripes':
        pattern = create_diagonal_stripes(baseline_image.shape, stripe_width=size, intensity=intensity)
        result_img = cv2.addWeighted(baseline_image, 1-intensity, pattern, intensity, 0)
    elif pattern_type == 'Dots':
        pattern = create_dot_pattern(baseline_image.shape, dot_spacing=size*2, dot_size=size//4, intensity=intensity)
        result_img = cv2.addWeighted(baseline_image, 1-intensity, pattern, intensity, 0)
    
    faces = detect_faces_cascade(result_img)
    result_img = draw_faces(result_img, faces)
    
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    plt.title(f'{pattern_type} Pattern - Faces Detected: {len(faces)}')
    plt.axis('off')
    plt.show()

## Part 4: Simple Defense Mechanisms

Now let's explore how to make face detection more robust.

In [ ]:
def preprocess_defensive(image):
    """
    Apply defensive preprocessing
    """
    # Method 1: Slight blur (removes high-frequency noise)
    blurred = cv2.GaussianBlur(image, (3, 3), 0)
    
    # Method 2: Median filter (removes salt-and-pepper noise)
    median = cv2.medianBlur(image, 3)
    
    # Method 3: Bilateral filter (preserves edges while removing noise)
    bilateral = cv2.bilateralFilter(image, 9, 75, 75)
    
    # Method 4: Morphological opening (removes small noise)
    kernel = np.ones((3,3), np.uint8)
    morph = cv2.morphologyEx(image, cv2.MORPH_OPEN, kernel)
    
    return blurred, median, bilateral, morph

def ensemble_detection(image, voting_threshold=2):
    """
    Use multiple parameter settings and vote
    """
    parameter_sets = [
        (1.05, 3),  # Very sensitive
        (1.1, 5),   # Normal
        (1.15, 6),  # Slightly conservative
        (1.2, 7),   # Conservative
    ]
    
    all_detections = []
    for scale, neighbors in parameter_sets:
        faces = detect_faces_cascade(image, scale, neighbors)
        all_detections.append(faces)
    
    # Simple voting: keep faces detected by at least 'voting_threshold' methods
    # (This is a simplified version - real implementation would check overlap)
    return all_detections[1]  # For now, just return normal settings

print("Defense mechanisms ready.")

### Task 4.1: Defense Evaluation

Test how preprocessing defenses work against noisy images.

In [ ]:
# Create an adversarial image
adversarial = add_noise(baseline_image, 'gaussian', 0.2)

# Apply defenses
blurred, median, bilateral, morph = preprocess_defensive(adversarial)

# Test detection on each
defenses = [
    ('Adversarial (No Defense)', adversarial),
    ('Gaussian Blur Defense', blurred),
    ('Median Filter Defense', median),
    ('Bilateral Filter Defense', bilateral),
    ('Morphological Defense', morph)
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Original clean image for reference
faces_clean = detect_faces_cascade(baseline_image)
axes[0].imshow(cv2.cvtColor(draw_faces(baseline_image, faces_clean), cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Clean Original\nFaces: {len(faces_clean)}')
axes[0].axis('off')

for i, (name, img) in enumerate(defenses):
    faces = detect_faces_cascade(img)
    result_img = draw_faces(img, faces)
    axes[i+1].imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    axes[i+1].set_title(f'{name}\nFaces: {len(faces)}')
    axes[i+1].axis('off')

plt.suptitle('Defensive Preprocessing Against Gaussian Noise (0.2 intensity)', fontsize=16)
plt.tight_layout()
plt.show()